# Week 2 · Logistic regression, evaluation, and k-nearest neighbors

The next two syllabus topics: **logistic regression** (the basic classifier) and **k-nearest neighbors**, plus how to **evaluate** a classifier. You did one logistic regression step by hand in [Diagnostic A ↗](https://claude.ai/artifact/XcTz8wtHd9iK9MnkYM68FF#week0~diag) on Oct 15. By the end of section 3 you'll know where every number in that answer came from. **Reading:** ISL Ch. 4.1–4.3 (logistic regression), Ch. 2.2.3 (kNN), Ch. 5.1 (cross-validation); StatQuest on logistic regression and on ROC/AUC.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

## Words you'll see
- **logit $z$**: The raw score before squashing: $z = w^\top x + b$, any real number.
- **sigmoid $\sigma$**: $\sigma(z) = \frac{1}{1 + e^{-z}}$, which squashes any number into $(0, 1)$.
- **decision boundary**: The set of points where the model is exactly 50/50, i.e. $\hat p = 0.5$.
- **positive / negative class**: The two labels, coded $y = 1$ and $y = 0$. "Positive" means the thing you're detecting (spam, defect, disease), not "good".
- **threshold**: The cut-off $t$: predict 1 if $\hat p \ge t$. The default is $t = 0.5$.

## 1. From a score to a probability: the sigmoid

**The sigmoid turns any score into a probability between 0 and 1.**

**The setting.** Binary classification: $y \in \{0, 1\}$. A linear score $z = w^\top x + b$ can be any number, but a probability must lie between 0 and 1. The sigmoid does the conversion:

$$
\hat p = P(y = 1 \mid x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = w^\top x + b
$$

| $z$ | −4 | −2 | −1 | 0 | 1 | 2 | 4 |
|---|---|---|---|---|---|---|---|
| $\sigma(z)$ | 0.018 | 0.119 | 0.269 | 0.500 | 0.731 | 0.881 | 0.982 |

**Where it comes from: log-odds.** Solve $\hat p = \sigma(z)$ for $z$ and you get $z = \log \frac{\hat p}{1 - \hat p}$. The ratio $\frac{\hat p}{1 - \hat p}$ is the **odds**. So logistic regression says that the **log-odds are linear in the features**: each unit increase in $x_j$ adds $w_j$ to the log-odds, i.e. multiplies the odds by $e^{w_j}$.

**Key properties.**

| Property | Formula | Why it matters |
|---|---|---|
| midpoint | $\sigma(0) = 0.5$ | the decision boundary is $z = 0$ |
| symmetry | $\sigma(-z) = 1 - \sigma(z)$ | $P(y=0) = \sigma(-z)$ |
| derivative | $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$ | makes the gradient below simple |
| range | $0 < \sigma(z) < 1$ | never exactly 0 or 1, so the log in the loss is safe in theory |

**The derivative, step by step.** Write $\sigma(z) = (1 + e^{-z})^{-1}$ and use the chain rule:

$$
\sigma'(z) = -(1 + e^{-z})^{-2} \cdot (-e^{-z}) = \frac{e^{-z}}{(1 + e^{-z})^2} = \underbrace{\frac{1}{1 + e^{-z}}}_{\sigma(z)} \cdot \underbrace{\frac{e^{-z}}{1 + e^{-z}}}_{1 - \sigma(z)}
$$

In [2]:
import numpy as np
sigmoid = lambda z: 1 / (1 + np.exp(-z))
print(sigmoid(np.array([-4, -2, -1, 0, 1, 2, 4])).round(3))   # → [0.018 0.119 0.269 0.5   0.731 0.881 0.982]
print(round(sigmoid(-1.3), 4), round(1 - sigmoid(1.3), 4))       # → 0.2142 0.2142   σ(-z) = 1 - σ(z)

[0.018 0.119 0.269 0.5   0.731 0.881 0.982]
0.2142 0.2142


## 2. The loss: binary cross-entropy

**Logistic regression is trained by minimizing binary cross-entropy, which is the negative log-likelihood of the labels.**

**Where it comes from.** Each label is a coin flip with probability $\hat p_i$ of being 1, so $P(y_i \mid x_i) = \hat p_i^{\,y_i} (1 - \hat p_i)^{1 - y_i}$ (when $y_i = 1$ this is $\hat p_i$, and when $y_i = 0$ it's $1 - \hat p_i$). Multiply over the samples, take the log, flip the sign, and average:

$$
L(w, b) = -\frac{1}{n} \sum_{i=1}^{n} \Big[ y_i \log \hat p_i + (1 - y_i) \log(1 - \hat p_i) \Big]
$$

This is the coin MLE from [Week 1 · math §4 ↗](../week1/03_math_essentials.ipynb) (7 heads in 10: set the derivative of the log-likelihood to 0 and get $\hat p = 0.7$), except that each sample now has its own probability $\hat p_i$, which depends on its features. [Notebook 02 §6 ↗](02_map_and_entropy_in_depth.ipynb) shows the term-by-term behaviour: confident and right costs almost nothing; confident and wrong costs a lot. Unlike linear regression there is **no closed-form solution**, but the loss is **convex** (bowl-shaped), so gradient descent finds the global minimum.

## 3. The gradient, every step

**Step 1, the loss for one sample:** $\ell = -[y \log \hat p + (1 - y) \log(1 - \hat p)]$ with $\hat p = \sigma(z)$.

**The gradient of the cross-entropy with respect to the logit is just $\hat p - y$: prediction minus truth.**

**Step 2, the derivative with respect to $\hat p$:**

$$
\frac{\partial \ell}{\partial \hat p} = -\frac{y}{\hat p} + \frac{1 - y}{1 - \hat p}
$$

**Step 3, multiply by $\frac{\partial \hat p}{\partial z} = \hat p (1 - \hat p)$** (the sigmoid derivative from section 1) and expand:

$$
\frac{\partial \ell}{\partial z} = \Big(-\frac{y}{\hat p} + \frac{1 - y}{1 - \hat p}\Big)\, \hat p (1 - \hat p) = -y(1 - \hat p) + (1 - y)\,\hat p = -y + y\hat p + \hat p - y\hat p = \hat p - y
$$

**Step 4, to the weights.** Since $z = w^\top x + b$, we have $\frac{\partial z}{\partial w} = x$ and $\frac{\partial z}{\partial b} = 1$. Averaged over the dataset:

$$
\nabla_w L = \frac{1}{n} \sum_i (\hat p_i - y_i)\, x_i = \frac{1}{n} X^\top (\hat p - y) \qquad \frac{\partial L}{\partial b} = \frac{1}{n} \sum_i (\hat p_i - y_i)
$$

Compare with linear regression, $\frac{2}{n} X^\top (\hat y - y)$: **the same shape, "features times error"**. Only the predictions changed, from $Xw$ to $\sigma(Xw)$.

**Worked example: [Diagnostic A ↗](https://claude.ai/artifact/XcTz8wtHd9iK9MnkYM68FF#week0~diag), revisited.** $w = (1, -2)$, $b = 0.5$, $x = (2, 1)$, $y = 1$, learning rate $\eta = 0.1$:

| Quantity | Calculation | Value |
|---|---|---|
| logit $z$ | $1 \cdot 2 + (-2) \cdot 1 + 0.5$ | 0.5 |
| $\hat p = \sigma(z)$ | $\frac{1}{1 + e^{-0.5}}$ | 0.6225 |
| loss | $-\ln 0.6225$ | 0.4741 |
| error $\hat p - y$ | $0.6225 - 1$ | −0.3775 |
| $\nabla_w$ | $-0.3775 \times (2, 1)$ | (−0.7551, −0.3775) |
| new $w$ | $(1, -2) - 0.1 \times (-0.7551, -0.3775)$ | (1.0755, −1.9622) |
| new $b$ | $0.5 - 0.1 \times (-0.3775)$ | 0.5378 |
| new $\hat p$ | $\sigma(1.0755 \cdot 2 - 1.9622 + 0.5378) = \sigma(0.7265)$ | 0.6740 |
| new loss | $-\ln 0.6740$ | 0.3945 (it went down) |

In [3]:
w = np.array([1.0, -2.0]); b = 0.5; x = np.array([2.0, 1.0]); y = 1
p = sigmoid(w @ x + b)
w_new = w - 0.1 * (p - y) * x
b_new = b - 0.1 * (p - y)
p_new = sigmoid(w_new @ x + b_new)
print(round(p, 4), w_new.round(4), round(b_new, 4), round(p_new, 4), round(-np.log(p_new), 4))
# → 0.6225 [ 1.0755 -1.9622] 0.5378 0.674 0.3945

0.6225 [ 1.0755 -1.9622] 0.5378 0.674 0.3945


**The decision boundary** is where $\hat p = 0.5$, i.e. $z = 0$: the line $w^\top x + b = 0$. For the weights above, that's $x_1 - 2x_2 + 0.5 = 0$. Logistic regression always draws a **straight** boundary (a hyperplane in more dimensions).

**More than two classes: softmax.** Give each class $k$ its own score $z_k = w_k^\top x + b_k$, turn the scores into probabilities with $\operatorname{softmax}(z)_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$ ([your Week 1 exercise 11 ↗](../week1/04_exercises.ipynb)), and train with cross-entropy $-\log \hat p_{\text{true}}$. For scores $(2, 1, 0.1)$ the probabilities are $(0.659, 0.242, 0.099)$.

**Drill (on paper).** $w = (0.5, 1)$, $b = -1$, $x = (2, 1)$, $y = 0$. Compute $z$, $\hat p$, the gradient with respect to $w$, and the new $w$ with $\eta = 0.5$.

<details><summary>Worked answer</summary>

$z = 1 + 1 - 1 = 1$, $\hat p = \sigma(1) = 0.731$. Error $\hat p - y = 0.731$. $\nabla_w = 0.731 \times (2, 1) = (1.462, 0.731)$. New $w = (0.5, 1) - 0.5 \times (1.462, 0.731) = (-0.231, 0.634)$. The label was 0, so the step lowers the score for this $x$.

</details>

![alt text](logreg_gd_step.png)

### One gradient descent step for logistic regression

**The situation.** Each sample is a point in feature space. Here there are 2 features, so it's a plane with axes $x_1, x_2$. We have one training point, $x = (2, 1)$, with true label $y = 0$. The current model $w = (0.5, 1)$, $b = -1$ defines a boundary line $w^\top x + b = 0$:

$$0.5\,x_1 + x_2 - 1 = 0 \quad\Longleftrightarrow\quad x_2 = 1 - 0.5\,x_1$$

On one side of the line $z > 0$, so $\hat p > 0.5$ and the model predicts 1. On the other side it predicts 0. The point sits on the wrong side. One gradient descent step ($\eta = 0.5$) fixes that. In a real dataset there would be many points of both labels and each step would average their pushes; one point makes the mechanics clear.

**1. Score**

$$z = w^\top x + b = 0.5(2) + 1(1) - 1 = 1$$

$z > 0$, so the point is on the "predict 1" side, and $z$ is its signed distance from the line, scaled by $\|w\|$.

**2. Probability**

$$\hat p = \sigma(1) = \frac{1}{1 + e^{-1}} \approx 0.731$$

The model says 73% chance of class 1, but the truth is 0: a confident mistake, with loss $-\log(1 - 0.731) \approx 1.31$.

**3. Gradient**

Since $\frac{\partial \ell}{\partial z} = \hat p - y = 0.731$,

$$\nabla_w \ell = (\hat p - y)\,x = 0.731 \cdot (2, 1) = (1.462,\ 0.731)$$

**4. Update**

$$w_{\text{new}} = w - \eta\,\nabla_w \ell = (0.5, 1) - 0.5\,(1.462, 0.731) = (-0.231,\ 0.635)$$

**What happened geometrically.** The update subtracted a multiple of $x$ from $w$, which lowers $w^\top x$ for this point. The boundary rotated: the old line runs down-right below the point, and the new line $-0.231\,x_1 + 0.635\,x_2 - 1 = 0$ tilts up-right and passes above it. Check the new score:

$$z_{\text{new}} = -0.231(2) + 0.635(1) - 1 = -0.828, \qquad \hat p_{\text{new}} = \sigma(-0.828) \approx 0.30$$

The model now gives 30% for class 1, so it correctly predicts 0, and the loss drops to about 0.36.

**The general rule.** For each sample, the step moves $w$ by $-\eta(\hat p - y)\,x$:

- **Label 0, predicted too high:** subtract a multiple of $x$, so the boundary moves to put $x$ on the 0 side.
- **Label 1, predicted too low:** add a multiple of $x$.
- **Already right and confident:** $\hat p - y \approx 0$, so the point barely moves the boundary.

This exercise only updates $w$. Normally $b$ updates too: $b_{\text{new}} = -1 - 0.5(0.731) = -1.366$, which shifts the line (rather than rotating it) further from the point and pushes $\hat p$ down to about 0.23.

## 4. Evaluating a classifier

**The confusion matrix counts the four kinds of right and wrong answer, and every classification metric is built from it.**

|  | predicted 1 | predicted 0 |
|---|---|---|
| **actually 1** | TP (true positive) | FN (false negative: missed it) |
| **actually 0** | FP (false positive: false alarm) | TN (true negative) |

| Metric | Formula | Answers the question |
|---|---|---|
| accuracy | $\frac{TP + TN}{n}$ | what fraction of all predictions were right? |
| precision | $\frac{TP}{TP + FP}$ | when it says positive, how often is it right? |
| recall (sensitivity, TPR) | $\frac{TP}{TP + FN}$ | of the real positives, how many did it catch? |
| $F_1$ | $\frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$ | one number that is high only if both are high |
| false positive rate (FPR) | $\frac{FP}{FP + TN}$ | of the real negatives, how many got a false alarm? |

**Worked example.** 200 parts, 60 really defective. The model catches $TP = 40$, misses $FN = 20$, raises $FP = 10$ false alarms, and correctly passes $TN = 130$:

| Metric | Calculation | Value |
|---|---|---|
| accuracy | $\frac{40 + 130}{200}$ | 0.850 |
| precision | $\frac{40}{40 + 10}$ | 0.800 |
| recall | $\frac{40}{40 + 20}$ | 0.667 |
| $F_1$ | $\frac{2 \cdot 0.8 \cdot 0.667}{0.8 + 0.667}$ | 0.727 |
| FPR | $\frac{10}{10 + 130}$ | 0.071 |

**Counting TP with NumPy masks**

In code, each count is two comparisons joined with `&`, then `.sum()`. With `y = [1, 1, 0, 0, 1, 0]` (truth) and `y_hat = [1, 0, 0, 1, 1, 0]` (prediction):

```text
position             0      1      2      3      4      5
y                    1      1      0      0      1      0
y_hat                1      0      0      1      1      0
y == 1             True   True   False  False  True   False
y_hat == 1         True   False  False  True   True   False
both (&)           True   False  False  False  True   False    → .sum() = 2 = TP
```

- Positions 0 and 4 are True in both rows: really 1 and predicted 1, so TP = 2.
- Position 3: really 0, predicted 1, so it's a false positive (FP = 1). Position 1: really 1, predicted 0, so it's a false negative (FN = 1).
- Positions 2 and 5: 0 and 0, so TN = 2. Check: $2 + 1 + 1 + 2 = 6$, every position is counted exactly once.

**Why accuracy can lie.** With 1000 parts of which only 10 are defective, a model that always says "fine" has accuracy $\frac{990}{1000} = 0.99$ and recall $0$: it never catches a single defect. **On imbalanced data, look at precision, recall and PR-AUC**, as you did for the peak detector in KAMP.

**The threshold trade-off.** Lowering the threshold $t$ catches more positives (recall up) but raises more false alarms (precision down). The **ROC curve** plots TPR against FPR for every threshold. Its area, **ROC-AUC**, is the probability that a random positive gets a higher score than a random negative: 1 is perfect, 0.5 is random guessing.

**AUC by hand.** Positives scored $(0.9, 0.7, 0.4)$, negatives scored $(0.6, 0.3, 0.2)$. Of the $3 \times 3 = 9$ positive–negative pairs, the positive wins in $3 + 3 + 2 = 8$ (only $0.4 < 0.6$ loses), so $\text{AUC} = \frac{8}{9} = 0.889$.

In [4]:
from sklearn.metrics import roc_auc_score
print(round(roc_auc_score([1, 1, 1, 0, 0, 0], [0.9, 0.7, 0.4, 0.6, 0.3, 0.2]), 3))   # → 0.889

0.889


**Cross-validation.** One train/test split gives one noisy estimate. **k-fold CV** splits the training data into $k$ parts, trains on $k - 1$ of them, tests on the one left out, repeats $k$ times, and averages the $k$ scores. You use it to choose settings like $\lambda$ or $k$ for kNN. The final test set still stays untouched until the very end.

> ⚠️ **Common mistake.** **Data leakage through preprocessing.** Fitting a scaler (or any step that looks at the data) on *all* the data, then splitting, lets test-set information into training. **Split first, then fit every preprocessing step on the training part only.** In [notebook 06 ↗](06_pandas_and_sklearn.ipynb), a scikit-learn `Pipeline` does this for you automatically.

**Drill (on paper).** $TP = 30$, $FP = 30$, $FN = 10$, $TN = 130$. Compute accuracy, precision, recall and $F_1$.

<details><summary>Worked answer</summary>

$n = 200$. Accuracy $\frac{160}{200} = 0.80$. Precision $\frac{30}{60} = 0.50$. Recall $\frac{30}{40} = 0.75$. $F_1 = \frac{2 \cdot 0.5 \cdot 0.75}{1.25} = 0.60$.

</details>

## 5. k-nearest neighbors

**k-nearest neighbors predicts by majority vote among the $k$ training points closest to the query.**

**The setting.** There is no training step at all: kNN stores the training set. To predict for a new point $q$: (1) compute its distance to every training point, (2) take the $k$ closest, (3) vote (classification) or average (regression). The usual distance is Euclidean:

$$
d(x, q) = \lVert x - q \rVert = \sqrt{\sum_{j=1}^{d} (x_j - q_j)^2}
$$

**Worked example.** Six training points in 2-D, query $q = (3.5, 3)$:

| Point | Coordinates | Class | $d(\cdot, q)$ | Rank |
|---|---|---|---|---|
| P3 | (2, 3) | A | $\sqrt{1.5^2 + 0^2} = 1.500$ | 1 |
| P4 | (5, 4) | B | $\sqrt{1.5^2 + 1^2} = 1.803$ | 2 |
| P5 | (4, 5) | B | $\sqrt{0.5^2 + 2^2} = 2.062$ | 3 |
| P2 | (2, 1) | A | $\sqrt{1.5^2 + 2^2} = 2.500$ | 4 |
| P1 | (1, 2) | A | $\sqrt{2.5^2 + 1^2} = 2.693$ | 5 |
| P6 | (6, 6) | B | $\sqrt{2.5^2 + 3^2} = 3.905$ | 6 |

| $k$ | Neighbors | Vote | Prediction |
|---|---|---|---|
| 1 | P3 | A | **A** |
| 3 | P3, P4, P5 | A, B, B | **B** |
| 5 | P3, P4, P5, P2, P1 | A, B, B, A, A | **A** |

**The choice of $k$ changes the answer.** A small $k$ follows every local wiggle (low bias, high variance: it overfits). A large $k$ smooths everything out (high bias, low variance). Choose $k$ with cross-validation, and use an odd $k$ for two classes to avoid ties.

**Why scaling matters.** Distances add up feature differences, so a feature with big numbers dominates. Here are the **same three people**, with income written two ways:

| Person | Age | Income ($) | Income ($1000s) |
|---|---|---|---|
| a | 25 | 50,000 | 50 |
| b | 60 | 51,000 | 51 |
| c | 27 | 56,000 | 56 |

| Units used | $d(a, b)$ | $d(a, c)$ | a's nearest neighbor |
|---|---|---|---|
| income in $ | 1000.6 | 6000.0 | b (income dominates; age is ignored) |
| income in $1000s | 35.0 | 6.3 | c (now age dominates) |
| standardized | 2.214 | 2.289 | b, and the same in **any** units |

The neighbors depend on the units you happened to record, which makes no sense. **Standardize the features before kNN**, so every feature counts on the same scale.

**Key properties.** No training cost, but each prediction compares against every training point, which is slow for big data. It works well in low dimensions but suffers from the **curse of dimensionality**: in many dimensions all points end up almost equally far apart, so "nearest" stops meaning much.

In [5]:
train = np.array([[1, 2], [2, 1], [2, 3], [5, 4], [4, 5], [6, 6]], dtype=float)
labels = np.array(["A", "A", "A", "B", "B", "B"])
q = np.array([3.5, 3.0])
d = np.linalg.norm(train - q, axis=1)          # broadcasting: one distance per row
order = np.argsort(d)                          # positions sorted from nearest to farthest
print(d.round(3))                              # → [2.693 2.5   1.5   1.803 2.062 3.905]
for k in [1, 3, 5]:
    votes = labels[order[:k]]
    print(k, votes, max(set(votes), key=list(votes).count))
# → 1 ['A'] A
#   3 ['A' 'B' 'B'] B
#   5 ['A' 'B' 'B' 'A' 'A'] A

[2.693 2.5   1.5   1.803 2.062 3.905]
1 ['A'] A
3 ['A' 'B' 'B'] B
5 ['A' 'B' 'B' 'A' 'A'] A


**What the three kNN lines do, array by array**

```text
train (6, 2)       q (2,)          train - q  (q is subtracted from every row)
[ 1  2 ]                           [ -2.5  -1.0 ]
[ 2  1 ]                           [ -1.5  -2.0 ]
[ 2  3 ]     -    [ 3.5  3 ]   =   [ -1.5   0.0 ]
[ 5  4 ]                           [  1.5   1.0 ]
[ 4  5 ]                           [  0.5   2.0 ]
[ 6  6 ]                           [  2.5   3.0 ]

norm(..., axis=1): one length per ROW → d = [ 2.693  2.5  1.5  1.803  2.062  3.905 ]   shape (6,)

argsort(d): the POSITIONS, ordered from the smallest d to the largest

    d by position:   0: 2.693   1: 2.5   2: 1.5   3: 1.803   4: 2.062   5: 3.905
    smallest first:  2 (1.5), 3 (1.803), 4 (2.062), 1 (2.5), 0 (2.693), 5 (3.905)
    argsort(d)    =  [2 3 4 1 0 5]
```

- Row 2 of `train - q` is $(-1.5, 0)$, so its length is $\sqrt{1.5^2 + 0^2} = 1.5$: the closest point.
- `argsort` doesn't return the distances; it returns **where** they are. `order[:3]` = `[2, 3, 4]` are the 3 nearest training rows.
- `labels[order[:3]]` looks up those rows' labels: A, B, B.

**Drill (on paper).** 1-D training data: $1 \to A$, $2 \to A$, $4 \to B$, $7 \to B$, $8 \to B$. Predict $x = 3$ with $k = 1$ and with $k = 3$.

<details><summary>Worked answer</summary>

Distances from 3: $2, 1, 1, 4, 5$. $k = 1$: there's a tie between the points at 2 (A) and 4 (B), both at distance 1. You need a tie rule, e.g. drop to a smaller $k$ or pick at random. Ties are one reason to prefer an odd $k$ and to check your data for duplicates. $k = 3$: the points at 2, 4 and 1 vote A, B, A, so the prediction is **A**.

</details>

---
Next: `06_pandas_and_sklearn.ipynb`, which runs all of this on real data with the libraries. Exercises 6–12 in `07_exercises.ipynb` cover this notebook.